In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('Housing.csv')
df.head()

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished


In [3]:
df.isnull().sum()

price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

In [8]:
X = df[['area','bedrooms','stories','parking']]
y = df['price']

In [9]:
#1 splited and scaled data 
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [10]:
# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns)

In [13]:
# 2. Implemented Forward Selection
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.feature_selection import SequentialFeatureSelector
lr = LinearRegression()
forward_features = SequentialFeatureSelector(lr, n_features_to_select='auto', direction='forward', cv=5)
forward_features.fit(X_train_scaled, y_train)

forward_selected = X.columns[forward_features.get_support()].tolist()
print("Selected Features:", forward_selected)

Selected Features: ['area', 'stories']


In [14]:
# 2. Implemented backward Selection
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.feature_selection import SequentialFeatureSelector
lr = LinearRegression()
backward_feat = SequentialFeatureSelector(lr, n_features_to_select='auto', direction='backward', cv=5)
backward_feat.fit(X_train_scaled, y_train)

backward_selected = X.columns[backward_feat.get_support()].tolist()
print("Selected Features:", backward_selected)

Selected Features: ['area', 'stories']


In [18]:
# Comparison between Forward and Backward Selection
print("Forward only:", set(forward_selected) - set(backward_selected))
print("Backward only:", set(backward_selected) - set(forward_selected))


Differences between Forward and Backward Selection:
Forward only: set()
Backward only: set()


In [15]:
# since both forward and backward are giving same output area,stories there is no difference

In [21]:
# 4. Train Lasso Regression with Alpha Tuning
ls = LassoCV(alphas=np.logspace(-3, 2, 100), cv=5, random_state=42)
ls.fit(X_train_scaled, y_train)

# Identify zero and non-zero coefficients
coef_series = pd.Series(ls.coef_, index=X.columns)
ls_selected = coef_series[coef_series != 0].index.tolist()
shrunk_to_zero = coef_series[coef_series == 0].index.tolist()

print("\n--- Lasso Regression ---")
print(f"Optimal Alpha (penalty): {ls.alpha_:.4f}")
print("Lasso Selected Features:", ls_selected)
print("Features Shrunk to Zero:", shrunk_to_zero)

print("\nCoefficient Values:\n", coef_series)



--- Lasso Regression ---
Optimal Alpha (penalty): 100.0000
Lasso Selected Features: ['area', 'bedrooms', 'stories', 'parking']
Features Shrunk to Zero: []

Coefficient Values:
 area        739568.465541
bedrooms    273270.079217
stories     530571.922342
parking     349826.669068
dtype: float64


In [ ]:
# no coefficients shrunks to zero 